In [4]:
import soundfile as sf
import io
import numpy as np
import base64
import requests
import subprocess
import librosa
from IPython.display import Audio

In [5]:
SAMPLE_RATE = 16000

In [6]:
def compress_to_opus(bytes: bytes) -> bytes:
  process = subprocess.Popen(
    ["ffmpeg", "-i", "pipe:0", "-c:a", "libopus", "-f", "opus", "pipe:1"],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE # subprocess.DEVNULL 하면 속도 조금 더 빨라짐
  )

  out, err = process.communicate(input=bytes)
  return out, err 

In [7]:
def np_to_wav(audio: np.ndarray, sample_rate:int) -> bytes:
  buffer = io.BytesIO()
  sf.write(buffer, audio, sample_rate, format='wav')
  return buffer.getvalue()

In [8]:
audio, sr = librosa.load(".data/news_with_english.mp3", sr=SAMPLE_RATE)

In [9]:
total_samples = len(audio)

segments = []
pos = 0
while pos < total_samples:
  rand_len = int(np.random.normal(loc=32000, scale=400))
  rand_len = np.clip(rand_len, 28000, 36000)
  end = min(pos + rand_len, total_samples)

  chunk = audio[pos:end]
  segments.append(chunk)
  pos = end

In [10]:
idx = 0

In [11]:
audio = segments[idx]
idx += 1
Audio(audio, rate=SAMPLE_RATE)

In [43]:
output = []
for segment in segments:
  audio = np_to_wav(segment, SAMPLE_RATE)
  audio, _ = compress_to_opus(audio)
  audio_b64 = base64.b64encode(audio).decode('utf-8')
  params = {
    "group": "1", 
    "user": "1", 
    "audio": audio_b64
  }

  res = requests.get("http://localhost:8000/whisper/stt_duration", params=params)
  d = res.json()
  print(d)
  output = [ *output, *d['completed'] ]

{'completed': [], 'candidate': []}
{'completed': [], 'candidate': [{'start': 0.0, 'end': 0.72, 'text': ' BBC', 'lang': 'ko'}, {'start': 0.72, 'end': 1.38, 'text': ' 생방송', 'lang': 'ko'}, {'start': 1.38, 'end': 1.88, 'text': ' 인터뷰', 'lang': 'ko'}, {'start': 1.88, 'end': 2.24, 'text': ' 도중에', 'lang': 'ko'}, {'start': 2.24, 'end': 2.98, 'text': ' 자녀', 'lang': 'ko'}, {'start': 2.98, 'end': 3.42, 'text': ' 난입', 'lang': 'ko'}, {'start': 3.42, 'end': 3.66, 'text': ' 사건.', 'lang': 'ko'}]}
{'completed': [], 'candidate': []}
{'completed': [], 'candidate': [{'start': 0.0, 'end': 0.72, 'text': ' BBC', 'lang': 'ko'}, {'start': 0.72, 'end': 1.38, 'text': ' 생방송', 'lang': 'ko'}, {'start': 1.3885625, 'end': 1.8885625, 'text': ' 인터뷰', 'lang': 'ko'}, {'start': 1.8885625, 'end': 2.2285625, 'text': ' 도중에', 'lang': 'ko'}, {'start': 2.2285625, 'end': 3.0085625, 'text': ' 자녀', 'lang': 'ko'}, {'start': 3.0085625, 'end': 3.4285625, 'text': ' 난입', 'lang': 'ko'}, {'start': 3.4285625, 'end': 4.0885625, 'text': ' 사건

In [44]:
for s in output:
  print(s['text'])

BBC 생방송 인터뷰 도중에 자녀 난입 사건으로 스타가 된 미국인 교수 가족이 오늘 카메라 앞에 섰습니다.
유튜브 스타가 된 4살짜리 딸은 이번엔 사탕을 입에 물고 등장했습니다.
배영진 기자입니다.
BBC 인터뷰 도중 딸과 아들의 등장으로 일약 스타가 된 로버트 켈리 부산대 교수 유튜브 영상 조회수가 1600만 건이 넘는데 켈리 교수 가족은 세계적인 유명인사가 됐습니다.
이렇게 언론의 관심이 커지자 켈리 교수 가족이 기자회견에 나섰습니다.
춤을 췄던 첫째 딸 메리아는 사탕을 물었고 둘째 아들 존은 엄마 품에 안긴 모습이었습니다. 
thought it was a disaster. 
I immediately called texted or or emailed the BBC. 
I communicated with the BBC immediately afterwards, and I apologized to them. 
I said that if they never called us back or never asked me to be on television again, I would understand.
귀여운 춤으로 화제가 된 딸에 대한 질문도 잇따랐습니다. 
That's She's four.
She has no idea.
cannot understand any of those.
당시 BBC와 인터뷰한 내용은 한국의 대통령 탄핵 사건이었습니다. 
months, six months, millions of people on the streets. 
no one's car got burned, the protesters even picked up their trash. 
I find that that's just a model of a 일부 외국 네티즌들이 켈리 교수의 한국의 아내를 보모로 오해한 것에 대해서는 오히려 의연하게 대처했습니다.
이제는 저희 나라도 마찬가지고 세계적으로 다문화 과정들이 많으니까 이런 계기로 인해서 조금 바뀌었으면 켈리 좋겠다.
교수는 아내와 2